# 10c -- MTMC Stages 4-5: Association + Evaluation

**Prerequisite**: attach **10b's output** as a data source:
`Add Data -> Kernel Output -> search "mtmc-10b-stage-3-faiss-indexing" -> add`

**This is the iteration loop** -- edit the tuning params cell and re-run in ~6 min.
No GPU needed, no data download, no model inference.

| Stage | What | Time |
|---|---|---|
| 4 | Cross-camera association (AQE + Louvain graph clustering) | ~5 min |
| 5 | Evaluation: IDF1, MOTA, HOTA | ~1 min |

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile
from pathlib import Path
import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

In [ ]:
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

try:
    import faiss; print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try: pip("faiss-gpu")
    except Exception: pip("faiss-cpu")

try:
    import trackeval; print("trackeval ok")
except ImportError:
    pip("git+https://github.com/JonathonLuiten/TrackEval.git")

pip("motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click",
    "numpy", "scipy", "pandas", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
                      cwd=str(PROJECT))
print("\n\u2713 Dependencies installed")

In [ ]:
FAILED = []
_checks = [
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("loguru", "loguru"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("trackeval", "trackeval"),
]
for label, mod in _checks:
    try:
        __import__(mod)
        print(f"  \u2713 {label}")
    except ImportError as e:
        print(f"  \u2717 {label}: {e}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED} -- fix pip installs above")
print("\n\u2713 All required modules importable")

## 2. Load Checkpoint from 10b

Finds `checkpoint.tar.gz` in `/kaggle/input/mtmc-10b-stage-3-faiss-indexing/` and extracts it.

In [ ]:
PREV_SLUG = "mtmc-10b-stage-3-faiss-indexing"
PREV_INPUT = Path("/kaggle/input") / PREV_SLUG

if not PREV_INPUT.exists():
    for p in Path("/kaggle/input").iterdir():
        if PREV_SLUG in p.name or "stage3" in p.name or "10b" in p.name:
            PREV_INPUT = p; break

cp = PREV_INPUT / "checkpoint.tar.gz"

# Fallback: download via Kaggle API if not found in mounted input
if not cp.exists():
    print(f"checkpoint.tar.gz not found at {cp} -- attempting API download")
    import subprocess as _sp
    _dl_dir = Path("/tmp/kaggle_dl")
    _dl_dir.mkdir(parents=True, exist_ok=True)
    _r = _sp.run(
        ["kaggle", "kernels", "output",
         "mrkdagods/mtmc-10b-stage-3-faiss-indexing",
         "--file", "checkpoint.tar.gz",
         "-p", str(_dl_dir)],
        capture_output=True, text=True
    )
    print(_r.stdout); print(_r.stderr)
    cp = _dl_dir / "checkpoint.tar.gz"

assert cp.exists(), f"checkpoint.tar.gz not found at {cp}"

EXTRACT_DIR = Path("/tmp/pipeline_run")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {cp.stat().st_size/1024**2:.1f} MB ...")
with tarfile.open(str(cp), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json") as f:
    meta = json.load(f)
RUN_NAME = meta["run_name"]
DATA_OUT = EXTRACT_DIR
RUN_DIR  = EXTRACT_DIR / RUN_NAME
# GT annotations are in the repo under data/raw/cityflowv2/*/gt/gt.txt
# They were force-committed despite data/ being gitignored.
_repo_gt = PROJECT / "data" / "raw" / "cityflowv2"
if any((_repo_gt / cam / "gt" / "gt.txt").exists()
       for cam in ["S01_c001", "S01_c002", "S01_c003",
                   "S02_c006", "S02_c007", "S02_c008"]):
    GT_DIR = str(_repo_gt)
    print(f"\u2713 GT annotations at {GT_DIR}")
else:
    GT_DIR = ""
    print("WARNING: GT files not found in repo — eval will skip metrics")
print(f"\u2713 Checkpoint extracted -- run: {RUN_NAME}")
for s in ["stage1", "stage2", "stage3"]:
    d = RUN_DIR / s
    if d.exists(): print(f"  {s}: {len(list(d.rglob('*')))} files")
print(f"  GT dir: {GT_DIR}")

## 3. Tuning Parameters

**Edit these values** then re-run the cells below (~6 min). No need to re-run 10a or 10b.

In [ ]:
# ---- Stage 4: Cross-camera association -------------------------------------
# AQE: k nearest neighbours for query expansion (higher = smoother features)
AQE_K             = 7     # v7: 7 (best from scan, was 5)

# Minimum cosine similarity to form an edge in the Louvain graph
SIM_THRESH        = 0.50  # v7: 0.50 (best from scan, was 0.35)

# Louvain resolution (higher = more, smaller clusters)
LOUVAIN_RES       = 0.70  # v7: 0.70 (best from scan, was 0.8)

# Weight of appearance vs. spatio-temporal score (0.0=ST only, 1.0=appear only)
APPEARANCE_WEIGHT = 0.70  # v7: 0.70 (best from scan)

# ---- Stage 5: Evaluation ----------------------------------------------------
# CityFlowV2 GT includes BOTH multi-cam (81 in S01, 130 in S02) AND
# single-cam (14 in S01, 15 in S02) vehicles — total 240 annotated vehicles.
# Setting True filters single-cam predictions, dropping 29 GT vehicles → lower
# IDF1.  Keep False for AIC-protocol comparison against published SOTA (84.1%).
MTMC_ONLY = False

print("Stage 4 params:")
print(f"  aqe_k={AQE_K}  sim_thresh={SIM_THRESH}  louvain_res={LOUVAIN_RES}  appearance_weight={APPEARANCE_WEIGHT}")
print(f"Stage 5: mtmc_only_submission={MTMC_ONLY}")

## 4. Run Stages 4-5

In [ ]:
os.chdir(str(PROJECT))
cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/cityflowv2.yaml",
    "--stages", "4,5",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    "--override", f"stage4.association.query_expansion.k={AQE_K}",
    "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
    "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
    "--override", f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
    "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
]
if GT_DIR:
    cmd += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]
else:
    print("WARNING: GT_DIR is empty — eval will skip metric computation")
print("CMD:", " ".join(str(c) for c in cmd))
print("=" * 70)
t0 = time.time()
r = subprocess.run(cmd, cwd=str(PROJECT))
print("=" * 70)
elapsed = time.time() - t0
if r.returncode != 0:
    print(f"\u2717 FAILED after {elapsed/60:.1f} min"); sys.exit(r.returncode)
print(f"\u2713 Stages 4-5 done in {elapsed/60:.1f} min")

## 5. Results

In [ ]:
stage5_dir = RUN_DIR / "stage5"

def _pct(v):
    return f"{v:.1%}" if isinstance(v, float) else str(v)

metrics_files = sorted(stage5_dir.glob("metrics_*.json")) if stage5_dir.exists() else []
if metrics_files:
    print("=" * 65)
    print("EVALUATION RESULTS")
    print("=" * 65)
    for mf in metrics_files:
        m = json.loads(mf.read_text())
        m = m.get("metrics", m)
        cam = mf.stem.replace("metrics_", "")
        idf1 = _pct(m.get("IDF1", m.get("idf1", "?")))
        mota = _pct(m.get("MOTA", m.get("mota", "?")))
        hota = _pct(m.get("HOTA", m.get("hota", "?")))
        idsw = m.get("ID_Sw", m.get("id_switches", "?"))
        print(f"  {cam:12s}  IDF1={idf1}  MOTA={mota}  HOTA={hota}  IDsw={idsw}")

for fname in ["summary.json", "evaluation_report.json"]:
    sf = stage5_dir / fname
    if sf.exists():
        s = json.loads(sf.read_text())
        print("-" * 65 + "\n  GLOBAL:")
        for k in ["IDF1","MOTA","HOTA","ID_Sw","idf1","mota","hota","id_switches","mtmc_idf1"]:
            v = s.get(k)
            if v is not None: print(f"    {k}: {_pct(v)}")
        break

if not metrics_files:
    print("No metrics files found -- check stage5 output dir:", stage5_dir)

## 6. Automated Parameter Scan (optional)

Runs stages 4-5 across a grid of parameter values and reports the best combination.
Each combination takes ~2 min → a 12-point scan takes ~24 min total.

**Comment out this cell if you just want the single run above.**

In [ ]:
# ============================================================
# SET SCAN_ENABLED = True to run the grid search.
# This will overwrite the single-run results above.
# ============================================================
SCAN_ENABLED = True

if SCAN_ENABLED:
    import itertools

    # Grid to search — comment out axes you don't want to vary
    # When appearance_w changes, spatiotemporal is auto-adjusted: st_w = 1 - app_w - 0.05
    # so the three vehicle weights always sum to 1.0 (hsv fixed at 0.05).
    scan_grid = {
        "sim_thresh":       [0.40, 0.45, 0.50, 0.55],   # v9: added 0.40 (cleaner tracklets may match at lower thresh)
        "louvain_res":      [0.60, 0.70, 0.80],          # v7: narrowed around best (0.70)
        "aqe_k":            [5, 7, 9],                   # v7: extended to test k=9
        "reranking":        [False],                     # v7: scan showed reranking=False is best
        "appearance_w":     [0.65, 0.70, 0.75],          # v7: sweep around best (0.70)
    }
    HSV_W_FIXED = 0.05  # fixed hsv weight; spatiotemporal = 1 - appearance - hsv

    keys   = list(scan_grid.keys())
    combos = list(itertools.product(*[scan_grid[k] for k in keys]))
    print(f"Scanning {len(combos)} combinations ...")

    results = []
    for combo in combos:
        params = dict(zip(keys, combo))
        rerank_tag = "rr1" if params["reranking"] else "rr0"
        scan_run = f"scan_{params['sim_thresh']}_{params['louvain_res']}_{params['aqe_k']}_{rerank_tag}"

        # Stage 4 reads stage1/stage2/stage3 from output_base/run_name/.
        # Symlink the upstream outputs so the scan sub-dir looks like a full run.
        scan_dir = DATA_OUT / scan_run
        scan_dir.mkdir(parents=True, exist_ok=True)
        for stage_sub in ("stage1", "stage2", "stage3"):
            src = DATA_OUT / RUN_NAME / stage_sub
            dst = scan_dir / stage_sub
            if src.exists() and not dst.exists():
                dst.symlink_to(src)

        # Compute spatiotemporal weight to maintain sum=1.0
        # hsv is fixed at 0.05; st = 1 - appearance - hsv
        st_w = round(1.0 - params["appearance_w"] - HSV_W_FIXED, 4)

        cmd_scan = [
            sys.executable, "scripts/run_pipeline.py",
            "--config", "configs/default.yaml",
            "--dataset-config", "configs/datasets/cityflowv2.yaml",
            "--stages", "4,5",
            "--override", f"project.run_name={scan_run}",
            "--override", f"project.output_dir={DATA_OUT}",
            "--override", f"stage4.association.query_expansion.k={params['aqe_k']}",
            "--override", f"stage4.association.graph.similarity_threshold={params['sim_thresh']}",
            "--override", f"stage4.association.graph.louvain_resolution={params['louvain_res']}",
            "--override", f"stage4.association.weights.vehicle.appearance={params['appearance_w']}",
            "--override", f"stage4.association.weights.vehicle.spatiotemporal={st_w}",
            "--override", f"stage4.association.reranking.enabled={str(params['reranking']).lower()}",
            "--override", "stage5.mtmc_only_submission=false",
        ]
        if GT_DIR:
            cmd_scan += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]
        t0 = time.time()
        r = subprocess.run(cmd_scan, cwd=str(PROJECT), capture_output=True)
        elapsed = time.time() - t0

        # Read metric from evaluation report
        report = DATA_OUT / scan_run / "stage5" / "evaluation_report.json"
        idf1 = mota = hota = 0.0
        if report.exists():
            rp = json.loads(report.read_text())
            m = rp.get("metrics", rp)
            idf1 = m.get("mtmc_idf1", m.get("IDF1", m.get("idf1", 0.0)))
            mota = m.get("MOTA", m.get("mota", 0.0))
            hota = m.get("HOTA", m.get("hota", 0.0))

        results.append({**params, "st_w": st_w, "IDF1": idf1, "MOTA": mota, "HOTA": hota, "time": elapsed})
        status = "OK" if r.returncode == 0 else "FAIL"
        print(f"  [{status}] {params} st_w={st_w:.2f} -> IDF1={idf1:.3f} MOTA={mota:.3f} HOTA={hota:.3f} ({elapsed/60:.1f} min)")

    # Sort by MTMC IDF1
    results.sort(key=lambda x: x["IDF1"], reverse=True)
    print("\n" + "=" * 80)
    print("SCAN RESULTS (sorted by MTMC IDF1)")
    print("=" * 80)
    header = f"{'sim':<6} {'res':<6} {'aqe':<5} {'rerank':<8} {'app_w':<7} {'st_w':<7} {'IDF1':>7} {'MOTA':>7} {'HOTA':>7}"
    print(header)
    for r2 in results:
        print(f"{r2['sim_thresh']:<6} {r2['louvain_res']:<6} {r2['aqe_k']:<5} "
              f"{str(r2['reranking']):<8} {r2['appearance_w']:<7} {r2.get('st_w',0.25):<7} "
              f"{r2['IDF1']:>7.3f} {r2['MOTA']:>7.3f} {r2['HOTA']:>7.3f}")
    best = results[0]
    print(f"\nBEST: sim={best['sim_thresh']} res={best['louvain_res']} aqe={best['aqe_k']} "
          f"reranking={best['reranking']} -> IDF1={best['IDF1']:.3f}")
else:
    print("Scan disabled. Set SCAN_ENABLED = True to run grid search.")